In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Start with an empty message list
messages = []
answer = ""
system = """
you have to write your answers and code as concise as possible
"""

while True:
    user_input = input("> ")
    print(">", user_input)

    # Add the initial user question
    add_user_message(messages, user_input)

    # Get Claude's response
    # answer = chat(messages, system=system)
    with client.messages.stream(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=system,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="")   # ver la respuesta en vivo
            # Send each chunk to your client
            # pass
    
        # Get the complete message for database storage
        final_message = stream.get_final_message()
        answer = final_message.content[0].text
    
    # Add Claude's response to the conversation history
    add_assistant_message(messages, answer)
    print()

    print("---")
    print(answer)
    print("---")

    


In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

In [ ]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())
clean_json

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short
""" 

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])
text = chat(messages)
text.strip()

In [ ]:
from IPython.display import Markdown

Markdown(text)